# Design: REST Table Gateway — Write / Action Endpoints

**Date:** 2026-06-03  
**Design epic:** `bd-d1ub`  
**Plan id:** `d1402bb3-302e-45a9-b5e7-dcbc826ffd82`  
**Status:** Draft — pending user review

## Summary

The REST table gateway (`crates/spur-notebook/rest-table-gateway`) and its DuckDB extension wrapper (`rest-table-gateway-ext`) are currently **read-only**. This design adds **write/modify support** — submitting a REST API request (`POST`/`PUT`/`PATCH`/`DELETE`) from SQL, modeled as a manifest-declared, typed **action table function**:

```sql
SELECT * FROM polymarket_place_order(token_id := '0xabc', price := 0.55, size := 10);
```

This is the industry-standard "POST-as-function" pattern (see Industry Validation). Literal `INSERT INTO` DML is explicitly **out of scope** — it requires a C++ `StorageExtension`, unreachable from the Rust `VTab` API.

## Approved decisions

| # | Decision | Choice |
|---|----------|--------|
| Q1 | HTTP verb scope | **Full `POST` / `PUT` / `PATCH` / `DELETE`** |
| Q2 | Manifest modeling | **New `[[action]]` section** (RPC-framed, separate from `[[table]]`) |
| Q3 | Arg → request mapping | **Per-arg `in` tag** `{ path \| body \| query }` (mirrors OpenAPI) |
| Q4 | Return shape | **Hybrid** — typed `columns` when declared, else generic `(http_status, body)` row |
| Q5 | Safety posture | **Full** — `allow_writes` gate + no auto-retry + pagination bypass + optional `Idempotency-Key` + `dry_run` |
| Arch | Trait threading | **Approach 2** — dedicated `async fn act(ActionRequest)` on `Adapter` (default "unsupported" impl) + `IoBridge` `Job::Scan \| Job::Act` |

Each decision is recorded as a `spur-audit v1` comment on epic `bd-d1ub`.

## Current architecture (read-only)

The gateway exposes data exclusively through DuckDB **table functions** (`VTab` `bind`/`init`/`func`), which have no insert/update/delete hooks. The `Adapter` trait is `name()` / `catalog()` / `async scan()`. `http.rs` is **GET-only**.

```mermaid
flowchart LR
  SQL["SELECT * FROM polymarket_markets()"] --> VTab["ApiTableVTab / ApiFunctionVTab<br/>(bind / init / func)"]
  VTab --> Bridge["IoBridge.call(adapter, ScanRequest)"]
  Bridge --> IOThread["rest-gateway-io thread<br/>(current-thread tokio)"]
  IOThread --> Scan["adapter.scan(req)"]
  Scan --> HTTP["http.rs fetch_rows<br/>GET only, paginated"]
  HTTP --> API[("REST API")]
  API --> Rows["Vec&lt;RecordBatch&gt;"]
  Rows --> VTab
  VTab --> Out["DuckDB result rows"]
```

**Key files:**

| File | Role |
|------|------|
| `rest-table-gateway/src/adapter/mod.rs` | `Adapter` trait, `ScanRequest`, `TableDef`, `TableKind` |
| `rest-table-gateway/src/adapter/http.rs` | GET fetch + pagination (`fetch_rows`, `apply_auth`) |
| `rest-table-gateway/src/adapter/manifest.rs` | TOML manifest model (`SourceCfg`, `TableCfg`, ...) |
| `rest-table-gateway/src/adapter/manifest_adapter.rs` | `ManifestAdapter` — builds catalog + scan from manifest |
| `rest-table-gateway/src/adapter/json_to_batch.rs` | JSON rows → Arrow `RecordBatch` |
| `rest-table-gateway/src/vtab/bridge.rs` | `IoBridge` — async IO thread, `call()` |
| `rest-table-gateway-ext/src/lib.rs` | Extension entrypoint, `register_adapter`, VTab impls |

## Goal & scope

### In scope
- Manifest-declared **action endpoints** for all four write verbs.
- Typed argument → request mapping (path / body / query).
- Hybrid response rendering (typed columns or generic status row).
- Full write-safety posture (gate, no-retry, idempotency, dry-run).

### Out of scope
- **Literal `INSERT INTO` DML** — needs a C++ `StorageExtension` (the Postgres/MySQL/SQLite scanner path); not reachable from the Rust `VTab` API.
- **OAuth2 write-scope token refresh** — handled by the separate in-flight OAuth2 refresh-grant epic; this design only *consumes* `ResolvedAuth`.
- **Transactions / multi-step sagas** — each action is a single request.

## Industry validation (researched 2026-06-03)

| Precedent | Evidence |
|-----------|----------|
| **`http_request` community extension** (maint. `onnimonni`) | Exposes `http_post` / `http_put` / `http_patch` / `http_delete` as **both scalar AND table functions**, incl. body support (`http_post_form`, `http_post_multipart`). Direct precedent for POST-as-table-function. |
| **ERPL** (commercial; SAP / M365 / Dynamics) | "Call any HTTP API from SQL — GET, POST, PUT, PATCH, DELETE with headers, basic/bearer auth." Claims **100+ production users**. |
| **DuckDB UDF API** | Explicit `side_effects` flag officially sanctions non-pure functions (scalar API). |
| **Postgres / MySQL / SQLite scanners** | Reference for true `INSERT INTO` via `StorageExtension` + `ATTACH ... (READ_ONLY)` — confirms DML is the C++-only path we are *not* taking. |

**Conclusion:** the function-based write path is a proven, shipping pattern. Our differentiator vs. the generic `http_post(url, body)` extensions: **typed, schema-validated, manifest-declared** endpoints.

## Target architecture (Approach 2)

Actions thread through a **dedicated `act()` method** on the `Adapter` trait and a new `Job::Act` variant on the `IoBridge`, keeping write semantics (no-retry, idempotency, dry-run, pagination-bypass) fully isolated from the read path.

```mermaid
flowchart LR
  SQL["SELECT * FROM<br/>polymarket_place_order(token_id := …, price := …, size := …)"] --> AVTab["ApiActionVTab<br/>(bind routes args by in-tag)"]
  AVTab -->|allow_writes gate| Bridge["IoBridge.call(Job::Act(ActionRequest))"]
  Bridge --> IOThread["rest-gateway-io thread"]
  IOThread --> Act["adapter.act(ActionRequest)"]
  Act --> Send["http.rs send_request<br/>POST/PUT/PATCH/DELETE<br/>no retry · no pagination"]
  Send --> API[("REST API")]
  API --> Resp["(status, JSON body)"]
  Resp --> Render{columns declared?}
  Render -->|yes| Typed["json_to_batch → typed rows"]
  Render -->|no| Generic["(http_status INT, body JSON) row"]
  Typed --> AVTab
  Generic --> AVTab
```

## Manifest — new `[[action]]` section

```toml
[source]
name = "polymarket"
base_url = "https://clob.polymarket.com"
allow_writes = true                  # SAFETY GATE — actions not registered without this

[[action]]
name = "place_order"
method = "POST"                      # POST | PUT | PATCH | DELETE
path = "/orders/{token_id}"          # {arg} segments filled from in="path" args
response_path = "$.order"            # optional, reuses table machinery
idempotency_header = "Idempotency-Key"   # optional
dry_run_arg = "dry_run"              # optional; when true → compose+return, don't send

[action.args]                        # per-arg `in` tag (Q3 = A)
token_id = { in = "path",  type = "Utf8",    required = true }
price    = { in = "body",  type = "Float64", required = true, json = "price" }
size     = { in = "body",  type = "Float64", required = true }
verbose  = { in = "query", type = "Boolean", required = false, param = "verbose" }

[action.columns]                     # optional (Q4 = C): typed response …
order_id = { json = "$.id",     type = "Utf8" }
status   = { json = "$.status", type = "Utf8" }
# … omit [action.columns] entirely → generic (http_status INT, body JSON) row
```

New Rust config types in `manifest.rs`: `ActionCfg`, `ArgCfg { in_: ArgLocation, ty, required, json, param }`, `ArgLocation { Path, Body, Query }`. `SourceCfg` gains `allow_writes: bool` (default `false`).

## Argument mapping (`in` = path | body | query)

Each DuckDB named arg is routed by its `in` tag at `bind` time. PUT/PATCH/DELETE rely on **path** args for the RESTful resource id; POST/PUT/PATCH carry **body** fields; **query** covers flags like `?dry_run=true`.

```mermaid
flowchart TD
  Args["DuckDB named args<br/>place_order(token_id, price, size, verbose)"] --> Bind["ApiActionVTab.bind()"]
  Bind --> Path["in = path → /orders/{token_id}"]
  Bind --> Body["in = body → JSON { price, size }"]
  Bind --> Query["in = query → ?verbose=true"]
  Path --> Req["ActionRequest"]
  Body --> Req
  Query --> Req
  Req --> Validate{"required args present?<br/>path placeholders filled?"}
  Validate -->|no| Err["bind error (fail fast, no request sent)"]
  Validate -->|yes| OK["ready for adapter.act()"]
```

## `http.rs` — method-aware send path

Add a `send_request` alongside the existing GET helpers. **No pagination loop, no retry** — both live only in the read path.

```rust
pub struct HttpAction<'a> {
    pub client: &'a Client,
    pub method: reqwest::Method,        // POST | PUT | PATCH | DELETE
    pub url: String,                    // base + path (placeholders already filled)
    pub query: Vec<(String, String)>,
    pub body: Option<serde_json::Value>,
    pub auth: &'a ResolvedAuth,
    pub idempotency_key: Option<(String, String)>,  // (header_name, value)
}

/// Sends exactly once. Returns (status, parsed-or-null body). 204/empty → Value::Null.
pub async fn send_request(a: &HttpAction<'_>) -> Result<(u16, serde_json::Value)>;
```

Reuses `apply_auth`. Non-2xx → `GatewayError::Http` carrying status + body snippet (callers must see *why* a write failed).

## Adapter trait + bridge changes (Approach 2)

```rust
// adapter/mod.rs
pub struct ActionRequest {
    pub name: String,
    pub method: String,
    pub path: String,                          // with {placeholders} filled
    pub query: Vec<(String, String)>,
    pub body: Option<serde_json::Value>,
    pub auth: ResolvedAuth,
    pub idempotency_key: Option<String>,
    pub dry_run: bool,
}

#[async_trait]
pub trait Adapter: Send + Sync {
    fn name(&self) -> &str;
    fn catalog(&self) -> Vec<TableDef>;
    async fn scan(&self, req: ScanRequest) -> Result<Vec<RecordBatch>>;
    // NEW — default impl keeps Polymarket / GraphQL / test adapters compiling unchanged
    async fn act(&self, _req: ActionRequest) -> Result<Vec<RecordBatch>> {
        Err(GatewayError::Adapter("this adapter does not support actions".into()))
    }
}

pub enum TableKind {
    Table,
    TableFunction { arg_names: Vec<String> },
    Action { method: String, arg_specs: Vec<ArgSpec> },   // NEW
}
```

```mermaid
sequenceDiagram
  participant D as DuckDB
  participant V as ApiActionVTab
  participant B as IoBridge
  participant T as io thread
  participant A as ManifestAdapter
  participant H as http.rs
  participant R as REST API
  D->>V: bind(named args)
  V->>V: route args by in-tag → ActionRequest
  D->>V: init()
  V->>B: call(Job::Act(req))
  B->>T: send job
  T->>A: act(req).await
  A->>H: send_request(method, url, body, query, auth, idem)
  H->>R: POST/PUT/PATCH/DELETE (once)
  R-->>H: status + body
  H-->>A: (status, Value)
  A->>A: render rows (typed | generic)
  A-->>T: Vec<RecordBatch>
  T-->>B: reply
  B-->>V: rows
  D->>V: func() → emit chunk
```

`bridge.rs`: `Job` becomes an enum `{ Scan(adapter, ScanRequest, reply), Act(adapter, ActionRequest, reply) }`; the IO loop matches and calls `scan`/`act` accordingly. `call_act()` mirrors `call()`.

## Extension registration (`ext/src/lib.rs`)

`register_adapter` gains a `TableKind::Action` arm registering a new `ApiActionVTab` (generalized from `ApiFunctionVTab`).

```mermaid
flowchart TD
  Cat["adapter.catalog()"] --> Loop{"for each TableDef"}
  Loop -->|Table| RT["register ApiTableVTab"]
  Loop -->|TableFunction| RF["register ApiFunctionVTab"]
  Loop -->|Action| RA["register ApiActionVTab"]
  RA --> NP["named_parameters() from [action.args]"]
  RA --> BD["bind(): route args by in-tag → ActionRequest"]
  RA --> IN["init(): bridge.call(Job::Act) → rows"]
  RA --> FN["func(): write_batch_rows (reused)"]
```

- `named_parameters()` is derived per-action from `arg_specs` (replacing the hard-coded `token_id`/`depth` list).
- `bind` maps each named param into path/body/query by its `ArgLocation`.
- `init` calls `bridge.call_act(...)`; `func` reuses the existing `write_batch_rows` renderer.

## Return shape — hybrid (Q4 = C)

```mermaid
flowchart TD
  Resp["HTTP response (status, body)"] --> Empty{"204 / empty body?"}
  Empty -->|yes, typed| Zero["zero rows"]
  Empty -->|yes, generic| StatusOnly["(http_status, NULL) row"]
  Empty -->|no| Q{"[action.columns] present?"}
  Q -->|yes| RP{"response_path set?"}
  RP -->|yes| Extract["json_path_get(body, response_path)"]
  RP -->|no| Whole["use body root"]
  Extract --> Typed["json_to_batch → declared columns"]
  Whole --> Typed
  Q -->|no| Generic["single row: (http_status INT, body JSON)"]
```

Reuses the existing `json_to_batch` + `response_path` machinery for the typed branch; the generic branch is a fixed 2-column schema so fire-and-confirm calls (and `DELETE` 204s) need no manifest schema.

## Write-safety posture (Q5 = C)

```mermaid
flowchart TD
  Load["Manifest load / register_adapter"] --> Gate{"source.allow_writes<br/>OR SPUR_REST_ALLOW_WRITES?"}
  Gate -->|no| Skip["[[action]] NOT registered<br/>(read-only — cannot mutate)"]
  Gate -->|yes| Reg["register ApiActionVTab"]
  Reg --> Call["action invoked"]
  Call --> Dry{"dry_run arg = true?"}
  Dry -->|yes| Compose["compose request,<br/>return as row, DO NOT send"]
  Dry -->|no| Idem{"idempotency_header set?"}
  Idem -->|yes| AddKey["attach Idempotency-Key"]
  Idem -->|no| NoKey["send as-is"]
  AddKey --> SendOnce["send ONCE — no auto-retry"]
  NoKey --> SendOnce
```

| Guardrail | Mechanism |
|-----------|-----------|
| **Opt-in gate** | `source.allow_writes` (+ `SPUR_REST_ALLOW_WRITES` env). Mirrors `allow_unsigned_extensions` / Postgres `READ_ONLY`. Read-only manifests *cannot register* an action. |
| **No auto-retry** | Structural — the action path has no retry loop. |
| **Pagination bypass** | Structural — actions never paginate. |
| **Idempotency** | Optional `Idempotency-Key` header neutralizes speculative/duplicate planner invocations. |
| **Dry-run** | Composes + returns the would-be request as a row without sending. |

## Execution-semantics caveat

DuckDB table functions are pulled lazily by the optimizer and may be invoked speculatively or skipped if the result is not consumed. The scalar `side_effects` flag is **not** available on the table-function (`VTab`) path.

**Mitigations baked into this design:**
- **Consumption contract** — document that actions must be consumed (`SELECT * FROM action(...)`), never buried in a prunable subquery.
- **Idempotency key** — duplicate firings collapse server-side.
- **`allow_writes` gate** — bounds blast radius to manifests that explicitly opted in.

If a future endpoint truly needs guaranteed per-row firing, the scalar-UDF escape hatch (`side_effects = true`) remains available as a follow-up.

## Task decomposition (for the plan DAG)

```mermaid
flowchart LR
  T1["T1 · manifest<br/>ActionCfg / ArgCfg + parse tests"] --> T3
  T2["T2 · http.rs<br/>send_request + tests"] --> T3
  T3["T3 · Adapter::act +<br/>ManifestAdapter impl + catalog"] --> T4["T4 · IoBridge<br/>Job::Scan | Job::Act"]
  T4 --> T5["T5 · extension<br/>TableKind::Action + ApiActionVTab<br/>+ arg binding + rendering"]
  T5 --> T7["T7 · wiremock E2E<br/>all 4 verbs via loaded ext"]
  T6["T6 · safety<br/>allow_writes gate · dry_run<br/>idempotency · no-retry"] -.spans.-> T1
  T6 -.spans.-> T2
  T6 -.spans.-> T5
```

| Task | File(s) | Depends on |
|------|---------|-----------|
| T1 manifest schema | `adapter/manifest.rs` | — |
| T2 http send path | `adapter/http.rs` | — |
| T3 adapter act + catalog | `adapter/mod.rs`, `adapter/manifest_adapter.rs` | T1, T2 |
| T4 bridge Act variant | `vtab/bridge.rs` | T3 |
| T5 extension registration | `rest-table-gateway-ext/src/lib.rs` | T4 |
| T6 safety wiring (cross-cutting) | T1 / T2 / T5 | — |
| T7 E2E (4 verbs) | `rest-table-gateway-ext/tests/load_extension_e2e.rs` | T5 |

## Testing strategy

| Level | Coverage |
|-------|----------|
| **Unit — manifest** | Parse `[[action]]` incl. each `in` location, `allow_writes` default false, missing-required arg, path placeholder validation. |
| **Unit — http.rs** | `send_request` per verb against `wiremock`: body serialization, query params, auth header, idempotency header, 204/empty → `Value::Null`, non-2xx → error with status. |
| **Unit — rendering** | Typed columns via `response_path`; generic `(http_status, body)` fallback; zero-row on empty typed response. |
| **Integration** | `ManifestAdapter::act` end-to-end against `wiremock`; `act()` on a non-supporting adapter returns the "unsupported" error. |
| **E2E (T7)** | Load the built `.duckdb_extension`, register a write manifest with `allow_writes = true`, exercise `POST`/`PUT`/`PATCH`/`DELETE` via SQL against `wiremock`; assert a read-only manifest does **not** register the action; assert `dry_run` sends nothing. |

## Future work (deferred)

- **`openapi-import` / `nango-import` generation** of `[[action]]` blocks (the OpenAPI `in: path|query|body` model maps 1:1 to our `in` tag).
- **OAuth2 write-scope refresh** — consumed here, owned by the in-flight refresh-grant epic.
- **Scalar `side_effects` action variant** for guaranteed per-row firing.
- **Batch / multi-row writes** (`INSERT INTO ... SELECT` semantics) and request batching.
- **GraphQL mutations** via the existing `graphql.rs` transport.

---

*Generated as the design spec for epic `bd-d1ub`. Next step: `writing-plans` → beads-backed implementation plan.*